# NSynth Melody Match Task

This notebook evaluates models on a zero-shot melody match task using NSynth validation clips. The task tests whether models can identify when two note sequences follow the same melodic pattern (same interval sequence) versus different patterns, even when transposed to different pitch ranges.

In [ ]:
import json
import math
import random
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple
import sys

import torch
import torchaudio
import torch.nn.functional as F
import pandas as pd

from IPython.display import Audio, display

%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import seaborn as sns
import numpy as np

import sys
sys.path.append('../')
sys.path.append('../lightning_scripts')
sys.path.append('../model_configs')
sys.path.append('../byol-a')


from tqdm.auto import tqdm
# Ensure tqdm works in notebooks
try:
    from IPython import get_ipython
    if 'IPKernelApp' in get_ipython().config:
        tqdm.pandas()  # Enable pandas integration if needed
except:
    pass

from lightning_scripts.nsynth_triplet_dataset import NsynthTripletDataset
from lightning_scripts.byola_lightning_module  import BYOLAModule

# Import figure_utils for standardized plotting
sys.path.append('../')
import figure_utils
from importlib import reload
reload(figure_utils)
from figure_utils import (
    normalize_model_name,
    build_model_palette,
    get_standard_hue_order,
    get_standard_base_colors,
    get_model_groups,
    model_label,
    plot_grouped_bars,
)
import yaml

## Load pretrained models

In [ ]:
# Device and target sample rate for generated clips
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from lightning_scripts.zero_shot_utils import get_cochdnn9_models, MODEL_SR

layer_out = "relu4"
models, model_name_map = get_cochdnn9_models(layer_out=layer_out, device=DEVICE)


In [ ]:
## Setup standardized plotting colors and model names
# Get standardized hue order and colors
hue_order = get_standard_hue_order()
base_colors = get_standard_base_colors()
hue_dict = build_model_palette(hue_order, base_colors)

# Get model groups for legend handling
supervised_names, ssl_names, scaled_ssl_names, byol_names = get_model_groups(hue_order)

# Normalize model names in our map
normalized_model_name_map = {k: normalize_model_name(v) for k, v in model_name_map.items()}

In [ ]:
from lightning_scripts.zero_shot_utils import encode_audio, distance_metrics_on_triplet


## Melody Match Task

This task tests whether models can identify when two note sequences follow the same melodic pattern. Anchor and positive share the same interval sequence (melody), while negative uses a different interval sequence, even though all three are transposed to different pitch ranges.

In [ ]:
# Instantiate dataset and dataloader
melody_df = NsynthTripletDataset(n_examples=5000, seed=42, min_midi=30, max_midi=90, experiment_type="melody_match", balance_eval=True)

def nsynth_triplet_collate(batch):
    clips, sr, triplet = batch[0]
    return clips, sr, triplet

melody_loader = torch.utils.data.DataLoader(
    melody_df, batch_size=1, shuffle=False, collate_fn=nsynth_triplet_collate
)
print(f"Triplet dataset ready: {len(melody_df)} examples at target_sr={melody_df.target_sr}")


all_results = []
for batch_idx, (clips, sr, triplet) in tqdm(enumerate(melody_loader), total=len(melody_loader), desc="Evaluating triplets"):
    for name, model in models.items():
        sr_val = int(sr)
        interval = triplet["anchor"]["interval"]
        negative_interval = triplet["negative"]["interval"]
        interval_diff = negative_interval - interval
        instrument = triplet["anchor"]["instrument"]

        metrics = distance_metrics_on_triplet(model, clips, sr_val, device=DEVICE, include_cosine=True)
        normalized_name = normalized_model_name_map.get(name, name)
        metrics.update({"model_name": normalized_name, "model_internal": name, "batch_idx": batch_idx, "interval": interval, "instrument": instrument, "negative_interval": negative_interval, "interval_diff": interval_diff}
                      
                      )
        all_results.append(metrics)
    if batch_idx % 100 == 0:
        print(f"On batch {batch_idx}")


melody_match_results_df = pd.DataFrame(all_results)
display(melody_match_results_df.head())

In [ ]:
metrics

In [ ]:
## write out results to csv
out_dir = Path("../results_dfs")
out_dir.mkdir(exist_ok=True, parents=True)
melody_match_results_df.to_csv(out_dir / f"zero_shot_nsynth_melody_match_results_{layer_out}_v2.csv", index=False)


## Plot Results

In [ ]:
melody_match_results_df

In [ ]:
# rename 'BYOL-A' to "byol-a"
melody_match_results_df['model_name'] = melody_match_results_df['model_name'].str.replace('BYOL-A', 'byol-a')


In [ ]:
# Plot aggregate results using standardized plotting format
plot_df = melody_match_results_df[melody_match_results_df['model_name'].isin(hue_order)].copy()

fig, ax = plt.subplots(1, 1, figsize=(4, 4))

_, handles, labels = plot_grouped_bars(
    ax,
    plot_df,
    value_col='judgement_pos_lt_neg',
    title=f'Melody match {layer_out}',
    ylabel='Prop. hit rate',
    error_type='sem',
    capsize=0,
    hue_order=hue_order,
    hue_dict=hue_dict,
)

random_weight_mean = melody_match_results_df[melody_match_results_df['model_name'].str.contains('random')].judgement_pos_lt_neg.mean()
ax.axhline(random_weight_mean, color='gray', linestyle='--', linewidth=1)
ax.set_ylim(0.5, 1.0)
ax.grid(True, axis="y", alpha=0.3)

# Build legend
legend_handles = []
legend_labels = []
for handle, label in zip(handles, labels):
    legend_handles.append(handle)
    legend_labels.append(label)

# Ensure scaled SSL palette entries appear in legend, even if missing from data
for model in scaled_ssl_names:
    label = model_label(model)
    if label not in legend_labels:
        color = hue_dict.get(model, '#444444')
        legend_handles.append(Patch(facecolor=color, edgecolor='black'))
        legend_labels.append(label)

if legend_handles:
    fig.legend(
        legend_handles,
        legend_labels,
        frameon=True,
        fontsize=8,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.05),
        ncol=4,
    )

plt.tight_layout()
plt.show()

In [ ]:
# Plot aggregate results using standardized plotting format
plot_df = melody_match_results_df[melody_match_results_df['model_name'].isin(hue_order)].copy()

fig, ax = plt.subplots(1, 1, figsize=(4, 4))

handles, labels = plot_grouped_bars(
    ax,
    plot_df,
    value_col='judgement_pos_lt_neg',
    title=f'Melody match {layer_out}',
    ylabel='P(pos_l2 < neg_l2)',
    hue_order=hue_order,
    hue_dict=hue_dict,
)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_ylim(0.4, 1.0)
ax.grid(True, axis="y", alpha=0.3)

# Build legend
legend_handles = []
legend_labels = []
for handle, label in zip(handles, labels):
    legend_handles.append(handle)
    legend_labels.append(label)

# Ensure scaled SSL palette entries appear in legend, even if missing from data
for model in scaled_ssl_names:
    label = model_label(model)
    if label not in legend_labels:
        color = hue_dict.get(model, '#444444')
        legend_handles.append(Patch(facecolor=color, edgecolor='black'))
        legend_labels.append(label)

if legend_handles:
    fig.legend(
        legend_handles,
        legend_labels,
        frameon=True,
        fontsize=8,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.05),
        ncol=4,
    )

plt.tight_layout()
plt.show()

## Load ResNet18 models from eval_dict and supervised models

In [ ]:
import pickle

# Load ResNet18 SSL models from eval_dict
eval_dict_path = Path("train_config_manifests/resnet18_barlow_equivariant_lmbda_search_eval_dict.pkl")
eval_dict = pickle.load(open(eval_dict_path, 'rb'))

# ResNet18 supervised models
resnet18_word_config = Path("model_configs/supervised_models/word_resnet18_MatchedDataset_LARS.yaml")
resnet18_audioset_config = Path("model_configs/supervised_models/audioset_resnet18_MatchedDataset_LARS.yaml")
resnet18_multitask_config = Path("model_configs/supervised_models/word_speaker_audioset_resnet18_MatchedDataset_shuffle_one_gpu_LARS.yaml")

# Load ResNet18 SSL models from eval_dict
resnet18_models = {}
resnet18_model_name_map = {}

for idx, config_path in eval_dict.items():
    config_path = Path(config_path)
    # Extract lambda value from config name for naming
    config_name = config_path.stem
    if 'eq_lmbda' in config_name:
        # Extract lambda value (format: eq_lmbda_5e-01 means 0.5)
        import re
        match = re.search(r'eq_lmbda_(\d+)e-(\d+)', config_name)
        if match:
            coeff = float(match.group(1))
            exp = float(match.group(2))
            lambda_val = coeff * (10 ** -exp)
            model_key = f"resnet18_ssl_λ={lambda_val}"
            resnet18_models[model_key] = get_model(config_path, layer_out="layer4")
            resnet18_model_name_map[model_key] = f"ResNet18 ssl λ={lambda_val}"
        else:
            # Fallback naming
            model_key = f"resnet18_ssl_{idx}"
            resnet18_models[model_key] = get_model(config_path, layer_out="layer4")
            resnet18_model_name_map[model_key] = f"ResNet18 ssl {config_name}"
    elif 'invariant_only' in config_name:
        model_key = "resnet18_ssl_invar"
        resnet18_models[model_key] = get_model(config_path, layer_out="layer4")
        resnet18_model_name_map[model_key] = "ResNet18 ssl invar"

# Load ResNet18 supervised models
resnet18_models["resnet18_word"] = get_model(resnet18_word_config, supervised=True, layer_out="layer4")
resnet18_model_name_map["resnet18_word"] = "ResNet18 supervised word"

resnet18_models["resnet18_audioset"] = get_model(resnet18_audioset_config, supervised=True, layer_out="layer4")
resnet18_model_name_map["resnet18_audioset"] = "ResNet18 supervised audioset"

resnet18_models["resnet18_multitask"] = get_model(resnet18_multitask_config, supervised=True, layer_out="layer4")
resnet18_model_name_map["resnet18_multitask"] = "ResNet18 supervised multi-task"

# Push all ResNet18 models to device
for model in resnet18_models.values():
    model.to(DEVICE)

print(f"Loaded {len(resnet18_models)} ResNet18 models")

## Evaluate ResNet18 models

In [ ]:
# Evaluate ResNet18 models on melody match task
resnet18_all_results = []
for batch_idx, (clips, sr, triplet) in tqdm(enumerate(melody_loader), total=len(melody_loader), desc="Evaluating ResNet18 triplets"):
    for name, model in tqdm(resnet18_models.items(), desc="Evaluating ResNet18 models", leave=False):
        sr_val = int(sr)
        interval = triplet["anchor"]["interval"]
        negative_interval = triplet["negative"]["interval"]
        interval_diff = negative_interval - interval
        instrument = triplet["anchor"]["instrument"]

        metrics = distance_metrics_on_triplet(model, clips, sr_val, device=DEVICE)
        normalized_name = normalize_model_name(resnet18_model_name_map.get(name, name))
        metrics.update({
            "model_name": normalized_name, 
            "model_internal": name, 
            "batch_idx": batch_idx, 
            "interval": interval, 
            "instrument": instrument, 
            "negative_interval": negative_interval, 
            "interval_diff": interval_diff
        })
        resnet18_all_results.append(metrics)
    

resnet18_melody_match_results_df = pd.DataFrame(resnet18_all_results)
display(resnet18_melody_match_results_df.head())

## Plot ResNet18 results

In [ ]:
# Plot ResNet18 aggregate results using standardized plotting format
resnet18_plot_df = resnet18_melody_match_results_df[resnet18_melody_match_results_df['model_name'].isin(hue_order)].copy()

fig, ax = plt.subplots(1, 1, figsize=(4, 4))

handles, labels = plot_grouped_bars(
    ax,
    resnet18_plot_df,
    value_col='judgement_pos_lt_neg',
    title='Melody match (ResNet18)',
    ylabel='P(pos_l2 < neg_l2)',
    hue_order=hue_order,
    hue_dict=hue_dict,
)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_ylim(0.4, 1.0)
ax.grid(True, axis="y", alpha=0.3)

# Build legend
legend_handles = []
legend_labels = []
for handle, label in zip(handles, labels):
    legend_handles.append(handle)
    legend_labels.append(label)

# Ensure scaled SSL palette entries appear in legend, even if missing from data
for model in scaled_ssl_names:
    label = model_label(model)
    if label not in legend_labels:
        color = hue_dict.get(model, '#444444')
        legend_handles.append(Patch(facecolor=color, edgecolor='black'))
        legend_labels.append(label)

if legend_handles:
    fig.legend(
        legend_handles,
        legend_labels,
        frameon=True,
        fontsize=8,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.05),
        ncol=4,
    )

plt.tight_layout()
plt.show()